# PARC2026 — 72c OpenVLA-OFT A100 batch probe

OpenVLA-OFT固定commitを専用Python 3.10/Torch 2.2環境で、validated selected-pool streaming bridgeから1 optimizer stepだけ動かします。effective batch 32の最大micro-batchを探索するprobe-only工程です。


In [ ]:
import os, subprocess
from pathlib import Path
from google.colab import drive, userdata

drive.mount('/content/drive')
if not os.environ.get('HF_TOKEN'):
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not os.environ.get('HF_TOKEN'):
    raise RuntimeError('HF_TOKEN is required; add it to Colab Secrets as HF_TOKEN')
ROOT = Path('/content/parc2026')
ROOT.mkdir(parents=True, exist_ok=True)
REPO = ROOT / 'py_AI_m3_probe'
PIN = '5baa5e3297ae476fe6f462ebcaac6d988f1a8e43'
URL = 'https://github.com/yu37330/py_AI.git'
if not (REPO / '.git').exists():
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError(f'Existing non-Git directory: {REPO}')
    subprocess.run(['git', 'clone', '--no-checkout', URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', PIN], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', '--force', PIN], check=True)
got = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
if got != PIN:
    raise RuntimeError(f'Repository pin mismatch: {got}')
print('72c probe code:', got, flush=True)
env = os.environ.copy(); env['PY_AI_REPO'] = str(REPO)
subprocess.run(['python', '-u', str(REPO / 'tools/colab/run_m3_openvla_batch_probe.py')], cwd=str(REPO), env=env, check=True)
print('=== 72c COMPLETE ===', flush=True)
print('Probe only. M3 benchmark training has NOT started.', flush=True)
